# F1 Qualifying Monte Carlo Simulation

## Setup

### Packages

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import fastf1

### Setting up FastF1

Set up F1 data cache (optional)

In [ ]:
import os

# Create cache directory if it doesn't exist
file_path = 'replace with your file path'
cache_dir = os.path.expanduser(file_path)
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

# Enable the cache
fastf1.Cache.enable_cache(cache_dir)

# Check cache is correct
print(fastf1.Cache)

## Functions

### Driver session stats

In [ ]:
def create_driver_stats(session):
    ''''
    Takes session as input.
    Filters to only competitive, legal laps (within 107% of fastest).
    Creates LapTimeSeconds column.
    Exports driver lap time mean, standard deviation and count.
    Drivers only with 1 lap in session have median std.
    '''

    laps = session.pick_quicklaps().loc[lambda df: ~df['Deleted']].copy()
    laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()

    driver_stats = laps.groupby('Driver')['LapTimeSeconds'].aggregate(['mean', 'std', 'count'])

    backup_std = driver_stats['std'].median()
    driver_stats['std'] = driver_stats['std'].fillna(backup_std)

    return driver_stats

### Simulate qualifying

In [ ]:
def simulate_qualifying(driver_stats, n_attemps=2):
    '''
    Takes driver stats contain name, mean and std as input.
    Default number of attempts is 2, as is typical in Q3.
    Uses normal distribution with mean and std to generate a lap time.
    Returns list of drivers sorted by position, P1 first.
    '''

    results = {}

    for driver in driver_stats.index:
        mean = driver_stats.loc[driver, 'mean']
        std = driver_stats.loc[driver, 'std']

        lap_times = np.random.normal(mean, std, n_attemps)
        best_lap = lap_times.min()

        results[driver] = best_lap

    sorted_drivers = sorted(results, key=results.get)

    return sorted_drivers

### Repeated qualifying sim

In [ ]:
def qualifying_MC(driver_stats, n=500):
    '''
    Takes driver stats contain name, mean and std as input, and number of simulations, default 500.
    Returns dictionary containing drivers as keys, and values as dictionaries with counts on positions.
    '''

    from collections import defaultdict

    # Automically create position with count 0 if not seen before, with fresh dictionary for each driver.
    position_counts = defaultdict(lambda: defaultdict(int))

    for i in range(n):
        result = simulate_qualifying(driver_stats)

        # Add count for that position for driver, adjust for 0 indexing.
        for position, driver in enumerate(result):
            position_counts[driver][position + 1] +=1

    return position_counts

### Calculate probabilities for positions

In [ ]:
def get_position_probability(position_counts, n=500):
    '''
    Takes position counts in and returns counts as probabilities.
    '''
    position_probabilities = {}

    for driver, positions in position_counts.items():
        position_probabilities[driver] = {position : count / n 
                                          for position, count in position_counts[driver].items()}
    
    return position_probabilities

### Parent function

In [ ]:
def monte_carlo_qualifying(gp, year, n=500):
    '''
    Input is GP name, year, and number of simulations n.
    Returns dictionary of drivers with inner dictionary of probabilities for each position.
    '''
    if n < 1:
        raise ValueError('n must be a positive integer.')
    
    session = fastf1.get_session(year, gp, 'Q')
    session.load()

    q1, q2, q3 = session.laps.split_qualifying_sessions()

    driver_stats = create_driver_stats(q3)

    position_counts = qualifying_MC(driver_stats, n)
    position_probabilities = get_position_probability(position_counts, n)

    df = pd.DataFrame.from_dict(position_probabilities, orient='index')

    df = df.fillna(0)
    df.index.name = 'Driver'
    df.columns.name = 'Position'
    df = df.reindex(sorted(df.columns), axis=1)

    return df